#Raw Citations

In [ ]:
import os
import json
import itertools
import csv
import shutil

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing"
RAW_OUTPUT = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/co_citations_raw.csv" # This file now acts as the primary checkpoint
CHECKPOINT_STATUS_FILE = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/processed_files.txt" # Tracks files processed for resuming
# ---------------------

def initialize_state(output_csv, status_file):
    """
    Checks for existing checkpoint files and determines the starting mode.
    Returns: set of processed filenames, and CSV open mode ('w' or 'a').
    """
    processed_files = set()
    write_mode = 'w'

    # 1. Check for previously processed files
    if os.path.exists(status_file):
        with open(status_file, "r") as f:
            processed_files = set(f.read().splitlines())

        # 2. If status file exists AND the main output CSV exists, resume by appending
        if os.path.exists(output_csv):
            write_mode = 'a'
            print(f"RESUMING: Found {len(processed_files)} files already processed. Appending to {output_csv}.")
        else:
            # Status file exists but the data is gone: treat as fresh start, but clean up orphaned status
            print(f"WARNING: Status file found, but {output_csv} is missing. Starting fresh.")
            # Clear the status so we don't skip files we haven't saved data for
            if os.path.exists(status_file):
                os.remove(status_file)
            processed_files = set()

    return processed_files, write_mode

def update_status(status_file, processed_filename):
    """Appends the filename to the status file."""
    with open(status_file, "a") as f:
        f.write(processed_filename + "\n")

# --- MAIN EXECUTION ---

print("--- 1/2: EXTRACTING RAW CO-CITATIONS WITH CHECKPOINTING ---")
# Initialize the state (resuming or starting new)
processed_files, csv_mode = initialize_state(RAW_OUTPUT, CHECKPOINT_STATUS_FILE)
header = ["paper1", "paper2", "citing_article"]

files_in_dir = os.listdir(DATA_DIR)
files_to_process = [
    filename for filename in files_in_dir
    if filename.endswith(".json") and filename not in processed_files
]

print(f"Found {len(files_to_process)} new files to process.")
print("-" * 50)

# Open the RAW_OUTPUT file ONCE. Data is written incrementally.
# This file is the primary raw data storage and checkpoint.
with open(RAW_OUTPUT, csv_mode, newline="", encoding="utf-8") as outfile:
    writer = csv.writer(outfile)

    # Write header ONLY if starting a new file ('w' mode)
    if csv_mode == 'w':
        writer.writerow(header)

    # Process files
    for i, filename in enumerate(files_to_process, 1):
        file_path = os.path.join(DATA_DIR, filename)
        print(f"Processing {i}/{len(files_to_process)}: {filename}")

        # --- Start File Read and Processing ---
        file_rows = []
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                articles = json.load(f)
        except Exception as e:
            print(f"Error reading or parsing {filename} → {e}")
            continue

        if isinstance(articles, dict):
            articles = [articles]
        elif not isinstance(articles, list):
            print(f"Unexpected format in {filename}")
            continue

        for art in articles:
            article_id = art.get("id", "").replace("https://openalex.org/", "")
            referenced_works = art.get("referenced_works", [])

            if not isinstance(referenced_works, list) or len(referenced_works) < 2:
                continue

            referenced_clean = [
                ref.replace("https://openalex.org/", "")
                for ref in referenced_works
                if isinstance(ref, str) and ref.startswith("https://openalex.org/")
            ]

            if len(referenced_clean) < 2:
                continue

            # Create co-citation pairs
            for p1, p2 in itertools.combinations(referenced_clean, 2):
                file_rows.append([p1, p2, article_id])
        # --- End File Read and Processing ---

        # 🚨 CHECKPOINT WRITE (DATA) 🚨
        if file_rows:
            writer.writerows(file_rows) # Immediate write to RAW_OUTPUT
            print(f"  → Wrote {len(file_rows)} raw co-citations.")

        # 🚨 CHECKPOINT WRITE (STATUS) 🚨
        # Mark file as complete only after successful write
        update_status(CHECKPOINT_STATUS_FILE, filename)
        print("  → Status file updated.")


print(f"\n✔ RAW CO-CITATIONS SAVED (and checkpointed) TO {RAW_OUTPUT}")



## ✅ Finalization and Cleanup

print("\n--- 2/2: FINALIZATION AND CLEANUP ---")

# Calculate the total number of JSON files in the directory
total_json_files = len([f for f in files_in_dir if f.endswith(".json")])
# Calculate the number of files we tried to process in total (resumed + new)
current_processed_count = len(processed_files) + len(files_to_process)

# Cleanup: Remove the status file only if ALL available JSON files were processed.
if current_processed_count >= total_json_files and os.path.exists(CHECKPOINT_STATUS_FILE):
    os.remove(CHECKPOINT_STATUS_FILE)
    print("✔ All files processed. Checkpoint status file removed.")
else:
    print(f"⚠️ Checkpoint status file ({CHECKPOINT_STATUS_FILE}) retained for next run.")

#Citations_Count

In [ ]:
import os
import json
import itertools
import csv
import pandas as pd
from google.colab import drive

# --- CONFIGURATION (Adjust Paths as needed) ---
DATA_DIR = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing"
# CSV where all raw pairs are written incrementally (for article-level safety)
CHECKPOINT_OUTPUT = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/co_citations_checkpoint.csv"
# Text file to track which JSON files have been fully processed (for file-level resumability)
PROGRESS_LOG = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/progress_log.txt"
# Final output file for the aggregated counts
COUNT_OUTPUT = "/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/co_citation_counts.csv"
# -----------------------------------------------

# Mount Drive (if not already done)
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

print("--- 1/2: EXTRACTING RAW CO-CITATIONS WITH ROBUST CHECKPOINTING ---")

# 1. LOAD COMPLETED FILES LIST for resumability
completed_files = set()
if os.path.exists(PROGRESS_LOG):
    with open(PROGRESS_LOG, 'r') as f:
        completed_files = set(f.read().splitlines())
    print(f"Loaded {len(completed_files)} files from progress log. Resuming work.")
else:
    print("No progress log found. Starting from the beginning.")

# 2. INITIALIZE CHECKPOINT CSV
write_header = not os.path.exists(CHECKPOINT_OUTPUT)

# Open both the checkpoint CSV and the progress log for appending
with open(CHECKPOINT_OUTPUT, "a", newline="", encoding="utf-8") as outfile, \
     open(PROGRESS_LOG, "a") as logfile:

    writer = csv.writer(outfile)
    if write_header:
        writer.writerow(["paper1", "paper2", "citing_article"])

    files_to_process = os.listdir(DATA_DIR)

    for filename in files_to_process:
        if not filename.endswith(".json"):
            continue

        # --- RESUMABILITY CHECK (Skip entire file if already logged) ---
        if filename in completed_files:
            continue

        file_path = os.path.join(DATA_DIR, filename)
        print(f"Processing: {filename}")

        # --- START FILE PROCESSING ---
        article_rows = []
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                articles = json.load(f)
        except Exception as e:
            print(f"Error reading or parsing {filename} → {e}. Skipping file.")
            continue # Skip to next file if parsing fails

        if isinstance(articles, dict):
            articles = [articles]
        elif not isinstance(articles, list):
            print(f"Unexpected format in {filename}. Skipping file.")
            continue

        for art in articles:
            article_id = art.get("id", "").replace("https://openalex.org/", "")
            referenced_works = art.get("referenced_works", [])

            if not isinstance(referenced_works, list) or len(referenced_works) < 2:
                continue

            referenced_clean = [
                ref.replace("https://openalex.org/", "")
                for ref in referenced_works
                if isinstance(ref, str) and ref.startswith("https://openalex.org/")
            ]

            # Deduplication Fix: Ensures unique references per article
            referenced_clean = list(set(referenced_clean))

            if len(referenced_clean) < 2:
                continue

            # Generate and normalize pairs, writing to disk immediately
            for p1, p2 in itertools.combinations(referenced_clean, 2):
                # Normalization Fix: Ensure consistent order (p1 < p2)
                normalized_pair = tuple(sorted((p1, p2)))

                # --- ARTICLE-LEVEL CHECKPOINTING ---
                # Write the pair immediately to disk. If crash occurs here,
                # only this single article's pairs might be lost, not the whole file.
                writer.writerow([normalized_pair[0], normalized_pair[1], article_id])

        # --- FILE-LEVEL CHECKPOINTING ---
        # Mark file as complete only after successfully processing all articles
        logfile.write(filename + '\n')
        print(f"-> Successfully Checkpointed: {filename}")

print(f"\n✔ RAW CO-CITATIONS CHECKPOINTED TO {CHECKPOINT_OUTPUT}")

# --- 2/2: AGGREGATE COUNTS FROM CHECKPOINT FILE ---
print("\n--- 2/2: CALCULATING FINAL CO-CITATION COUNTS ---")

try:
    # Load the checkpoint file from disk
    # The 'paper1' and 'paper2' columns are already normalized (sorted)
    df_pairs = pd.read_csv(CHECKPOINT_OUTPUT)

    # Group by the pair and count the occurrences (vectorized and fast)
    co_citation_counts = df_pairs.groupby(['paper1', 'paper2']).size().reset_index(name='count')

    # Save the final results
    co_citation_counts.to_csv(COUNT_OUTPUT, index=False)

    print(f"\n✔ FINAL CO-CITATION COUNTS SAVED TO {COUNT_OUTPUT}")
    print(f"   Total unique co-citation pairs found: {len(co_citation_counts)}")

except Exception as e:
    print(f"An error occurred during final aggregation. Check if the checkpoint file is corrupted: {e}")